# Challenge: Asistente de Estatus de Pedidos
## WhatsApp Cloud API + Ollama (Llama 3.1)

Challenge del Hands-On de **WhatsApp Cloud API**. El asistente responde el estatus de pedidos por WhatsApp y usa **Ollama** como motor de lenguaje.

Mejoras respecto a la clase:
1. **Mensaje de bienvenida** para usuarios nuevos.
2. **Tono de respuesta controlado** (maximo 2-3 lineas, amable).

> En este repositorio la prueba local corre en el **PC local** (`127.0.0.1:11434`) con un servidor local de la Graph API de Meta, para no depender de credenciales reales. Este notebook replica el flujo para Colab.

## 1. Instalar y arrancar Ollama

Ollama no viene preinstalado en la VM de Colab: se instala y se arranca en segundo plano.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5 && curl http://localhost:11434/api/version

In [ ]:
# Descargar el modelo. En el challenge: llama3.1:8b
# En el PC local ya estan instalados qwen3.5:0.8b, gemma4:12b, granite4.2:8b, gpt-oss:20b
!ollama pull llama3.1:8b
!ollama list

## 2. Leer credenciales desde Colab Secrets

Panel de Secrets (icono de llave) -> `WHATSAPP_TOKEN`, `WHATSAPP_PHONE_ID`, `VERIFY_TOKEN`.

In [ ]:
from google.colab import userdata

WHATSAPP_TOKEN = userdata.get('WHATSAPP_TOKEN')
PH_NUMBER_ID = userdata.get('WHATSAPP_PHONE_ID')
VERIFY_TOKEN = userdata.get('VERIFY_TOKEN')

print('Token cargado:', bool(WHATSAPP_TOKEN))
print('Phone ID cargado:', bool(PH_NUMBER_ID))

## 3. Configurar el servidor del webhook

Estructuras en memoria: `mensajes_recibidos` (bitacora de payloads) y `usuarios_conocidos` (`set`).

In [ ]:
!pip install -q fastapi "uvicorn[standard]" requests pyngrok

from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import JSONResponse, PlainTextResponse
from typing import Set, List, Dict, Any

app = FastAPI(title='Asistente de Estatus de Pedidos')

mensajes_recibidos: List[Dict[str, Any]] = []
usuarios_conocidos: Set[str] = set()

OLLAMA_URL = 'http://localhost:11434/api/chat'
LLAMA_MODEL = 'llama3.1:8b'   # en el PC: qwen3.5:0.8b / gemma4:12b

SALUDO = ('¡Hola! 👋 Bienvenido/a al Asistente de Estatus de Pedidos. '
          'Dime el número de pedido (ejemplo: PED-1001) y te doy su estatus. 📦')

PEDIDOS = {
    'PED-1001': {'estado': 'En preparación', 'entrega': '2026-10-31', 'rastreo': 'MX-TRACK-889201'},
    'PED-1002': {'estado': 'En camino', 'entrega': '2026-10-30', 'rastreo': 'MX-TRACK-771044'},
    'PED-1003': {'estado': 'Entregado', 'entrega': '2026-10-18', 'rastreo': 'MX-TRACK-550912'},
}

## 4. Normalización de número mexicano

In [ ]:
import re

def normalizar_numero_mx(telefono: str) -> str:
    numero = re.sub(r'[^0-9+]', '', str(telefono).strip())
    if numero.startswith('00'):
        numero = '+' + numero[2:]
    if numero.startswith('+'):
        numero = numero[1:]
    if numero.startswith('0'):
        numero = '55' + numero[1:]
    if len(numero) == 12 and numero.startswith('52'):
        numero = '55' + numero[2:]
    if len(numero) == 10 and numero.isdigit():
        numero = '55' + numero
    return numero

print(normalizar_numero_mx('+52 1 55 1234 5678'))  # 5215512345678

## 5. Generación de respuesta con tono controlado

In [ ]:
import requests

PROMPT_SISTEMA = '''Eres el Asistente de Estatus de Pedidos de una tienda en línea.
Reglas de tono (obligatorias):
- Responde en ESPAÑOL, amable, en máximo 2-3 líneas.
- Ve directo al grano, sin preámbulos.
- Si piden estatus sin número, pídelo (ejemplo: PED-1001).
- No inventes estados, fechas ni rastreos.
- Máximo un emoji por respuesta.'''

def generar_respuesta(mensaje: str) -> str:
    payload = {
        'model': LLAMA_MODEL,
        'messages': [
            {'role': 'system', 'content': PROMPT_SISTEMA},
            {'role': 'user', 'content': mensaje},
        ],
        'stream': False,
        'options': {'temperature': 0.3, 'num_predict': 200},
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=60)
    r.raise_for_status()
    data = r.json()
    return (data.get('message', {}).get('content') or '').strip()

print(generar_respuesta('¿Qué métodos de pago aceptan?'))

## 6. Endpoints del webhook

In [ ]:
@app.get('/webhook')
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get('hub.mode') == 'subscribe' and params.get('hub.verify_token') == VERIFY_TOKEN:
        return PlainTextResponse(content=params.get('hub.challenge', ''), status_code=200)
    return JSONResponse(content={'error': 'Token incorrecto'}, status_code=403)

@app.post('/webhook')
async def recibir_mensajes(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
        value = body['entry'][0]['changes'][0]['value']
        mensajes = value.get('messages', [])
    except (KeyError, IndexError):
        return JSONResponse(content={'status': 'ignored'}, status_code=200)
    if not mensajes:
        return JSONResponse(content={'status': 'ignored'}, status_code=200)
    m = mensajes[0]
    telefono = normalizar_numero_mx(m['from'])
    texto = (m.get('text') or {}).get('body', '')
    background_tasks.add_task(procesar_mensaje, telefono, texto)
    return JSONResponse(content={'status': 'received'}, status_code=200)

## 7. Envío de la respuesta y saludo inicial

In [ ]:
GRAPH = 'https://graph.facebook.com/v19.0'

def enviar_mensaje_whatsapp(destinatario: str, mensaje: str) -> dict:
    url = f'{GRAPH}/{PH_NUMBER_ID}/messages'
    headers = {'Authorization': f'Bearer {WHATSAPP_TOKEN}', 'Content-Type': 'application/json'}
    data = {'messaging_product': 'whatsapp', 'to': destinario,
            'type': 'text', 'text': {'preview_url': False, 'body': mensaje}}
    r = requests.post(url, json=data, headers=headers, timeout=15)
    return {'ok': r.status_code == 200, 'http': r.status_code, 'respuesta': r.text[:300]}

def procesar_mensaje(telefono: str, texto: str) -> dict:
    es_nuevo = telefono not in usuarios_conocidos
    if es_nuevo:
        usuarios_conocidos.add(telefono)
        respuesta = SALUDO
    elif 'PED-' in texto.upper():
        folio = texto.upper()[texto.upper().index('PED-'):][:8].strip()
        pedido = PEDIDOS.get(folio)
        respuesta = (f'📦 {folio} — Estado: {pedido["estado"]}. '
                     f'Entrega estimada: {pedido["entrega"]}. Rastreo: {pedido["rastreo"]}.') if pedido \
                    else f'No encuentro el pedido {folio}. Revisa el numero, por favor. 🤔'
    else:
        respuesta = generar_respuesta(texto)
    envio = enviar_mensaje_whatsapp(telefono, respuesta)
    print(f'[{telefono}] {texto} -> {respuesta[:80]} | envio={envio["ok"]}')
    return envio

## 8. Exponer el servidor con ngrok y probar el flujo

In [ ]:
import threading, time, uvicorn
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel = ngrok.connect(8000)
print('URL publica:', tunel.public_url)
print('Callback URL en Meta:', tunel.public_url + '/webhook')

def correr():
    uvicorn.run(app, host='0.0.0.0', port=8000)

threading.Thread(target=correr, daemon=True).start()
time.sleep(3)
print('Servidor levantado')

In [ ]:
def payload(telefono: str, texto: str) -> dict:
    return {'object': 'whatsapp_business_account',
            'entry': [{'changes': [{'value': {'messaging_product': 'whatsapp',
                    'contacts': [{'wa_id': telefono}],
                    'messages': [{'from': telefono, 'type': 'text',
                                  'text': {'body': texto}}]}}]}]}

URL = tunel.public_url + '/webhook'

# 1) Usuario nuevo -> debe recibir el saludo
print(requests.post(URL, json=payload('5215512345678', 'Hola, buenas tardes'), timeout=30).text)
time.sleep(8)

# 2) Usuario conocido -> debe recibir la respuesta de Llama
print(requests.post(URL, json=payload('5215512345678', '¿Cuál es el estatus de mi pedido PED-1001?'), timeout=30).text)
time.sleep(15)

print('Bitacora local:', len(mensajes_recibidos), 'mensajes |', len(usuarios_conocidos), 'usuarios conocidos')